In [52]:
import pandas as pd
from pathlib import Path


### Combining enrolment data
df1 = pd.read_csv("data/raw/api_data_aadhar_enrolment/api_data_aadhar_enrolment/api_data_aadhar_enrolment_0_500000.csv", parse_dates=["date"], dayfirst=True)
df2 = pd.read_csv("data/raw/api_data_aadhar_enrolment/api_data_aadhar_enrolment/api_data_aadhar_enrolment_500000_1000000.csv", parse_dates=["date"], dayfirst=True)
df3 = pd.read_csv("data/raw/api_data_aadhar_enrolment/api_data_aadhar_enrolment/api_data_aadhar_enrolment_1000000_1006029.csv", parse_dates=["date"], dayfirst=True)

# Stack rows
df = pd.concat([df1, df2, df3], ignore_index=True)

# Save
df.to_csv("data/combined/aadhar_enrolment_combined.csv", index=False)

In [53]:
### Combining demographic data

df1 = pd.read_csv("data/raw/api_data_aadhar_demographic/api_data_aadhar_demographic/api_data_aadhar_demographic_0_500000.csv", parse_dates=["date"], dayfirst=True)
df2 = pd.read_csv("data/raw/api_data_aadhar_demographic/api_data_aadhar_demographic/api_data_aadhar_demographic_500000_1000000.csv", parse_dates=["date"], dayfirst=True)
df3 = pd.read_csv("data/raw/api_data_aadhar_demographic/api_data_aadhar_demographic/api_data_aadhar_demographic_1000000_1500000.csv", parse_dates=["date"], dayfirst=True)
df4 = pd.read_csv("data/raw/api_data_aadhar_demographic/api_data_aadhar_demographic/api_data_aadhar_demographic_1500000_2000000.csv", parse_dates=["date"], dayfirst=True)
df5 = pd.read_csv("data/raw/api_data_aadhar_demographic/api_data_aadhar_demographic/api_data_aadhar_demographic_2000000_2071700.csv", parse_dates=["date"], dayfirst=True)

# Stack rows
df = pd.concat([df1, df2, df3, df4, df5], ignore_index=True)

# Save
df.to_csv("data/combined/aadhar_demographic_combined.csv", index=False)

In [ ]:
### Combining biometric data

df1 = pd.read_csv("data/raw/api_data_aadhar_biometric/api_data_aadhar_biometric/api_data_aadhar_biometric_0_500000.csv",parse_dates=["date"], dayfirst=True)
df2 = pd.read_csv("data/raw/api_data_aadhar_biometric/api_data_aadhar_biometric/api_data_aadhar_biometric_500000_1000000.csv",parse_dates=["date"], dayfirst=True)
df3 = pd.read_csv("data/raw/api_data_aadhar_biometric/api_data_aadhar_biometric/api_data_aadhar_biometric_1000000_1500000.csv",parse_dates=["date"], dayfirst=True)
df4 = pd.read_csv("data/raw/api_data_aadhar_biometric/api_data_aadhar_biometric/api_data_aadhar_biometric_1500000_1861108.csv",parse_dates=["date"], dayfirst=True)

# Stack rows
df = pd.concat([df1, df2, df3, df4], ignore_index=True)

# Save
df.to_csv("data/combined/aadhar_biometric_combined.csv", index=False)

### 1. Cleaning the dataset

In [63]:
# 1.1 Load the csv files
import pandas as pd

enroll = pd.read_csv("data/combined/aadhar_enrolment_combined.csv", parse_dates=["date"], dayfirst=False)
demo = pd.read_csv("data/combined/aadhar_demographic_combined.csv", parse_dates=["date"], dayfirst=False)
bio = pd.read_csv("data/combined/aadhar_biometric_combined.csv", parse_dates=["date"], dayfirst=False)


In [64]:
# 1.2 Inspect schemas
print(enroll.columns)
print(demo.columns)
print(bio.columns)


Index(['date', 'state', 'district', 'pincode', 'age_0_5', 'age_5_17',
       'age_18_greater'],
      dtype='object')
Index(['date', 'state', 'district', 'pincode', 'demo_age_5_17',
       'demo_age_17_'],
      dtype='object')
Index(['date', 'state', 'district', 'pincode', 'bio_age_5_17', 'bio_age_17_'], dtype='object')


In [65]:
# 1.3 Standardize date columns
for df in [enroll, demo, bio]:
    df["date"] = pd.to_datetime(df["date"], dayfirst=False, errors="coerce")


In [66]:
assert enroll["date"].isna().sum() == 0
assert demo["date"].isna().sum() == 0
assert bio["date"].isna().sum() == 0

In [67]:
print(pd.api.types.is_datetime64_any_dtype(enroll["date"]))
print(pd.api.types.is_datetime64_any_dtype(demo["date"]))
print(pd.api.types.is_datetime64_any_dtype(bio["date"]))

True
True
True


In [68]:
# 1.4 Standardize geographic keys

for df in [enroll, demo, bio]:
    df["state"] = (
        df["state"]
        .str.strip()
        .str.upper()
    )

    df["district"] = (
        df["district"]
        .str.strip()
        .str.upper()
    )

    df["pincode"] = df["pincode"].astype(str).str.zfill(6)



In [69]:
# 1.5 Ensure numeric columns are numeric

numeric_cols = {
    "enroll": ["age_0_5", "age_5_17", "age_18_greater"],
    "demo":   ["demo_age_5_17", "demo_age_17_"],
    "bio":    ["bio_age_5_17", "bio_age_17_"],
}

for col in numeric_cols["enroll"]:
    enroll[col] = pd.to_numeric(enroll[col], errors="coerce")

for col in numeric_cols["demo"]:
    demo[col] = pd.to_numeric(demo[col], errors="coerce")

for col in numeric_cols["bio"]:
    bio[col] = pd.to_numeric(bio[col], errors="coerce")


In [ ]:
assert enroll[numeric_cols["enroll"]].isna().sum().sum() == 0
assert demo[numeric_cols["demo"]].isna().sum().sum() == 0
assert bio[numeric_cols["bio"]].isna().sum().sum() == 0

In [71]:
enroll.to_csv("data/cleaned/clean_enrolment_raw.csv", index=False)
demo.to_csv("data/cleaned/clean_demographic_raw.csv", index=False)
bio.to_csv("data/cleaned/clean_biometric_raw.csv", index=False)


### 2. Aggregating the dataset

In [ ]:
# 2.1 Aggregate Aadhaat enrolment data

enroll_dist = (
    enroll
    .groupby(["date", "state", "district"], as_index=False)
    .agg({
        "age_0_5": "sum",
        "age_5_17": "sum",
        "age_18_greater": "sum"
    })
)

In [75]:
enroll_dist.head(10)

,date,state,district,age_0_5,age_5_17,age_18_greater
0,2025-03-02,MEGHALAYA,EAST KHASI HILLS,11,61,37
1,2025-03-09,BIHAR,BHAGALPUR,13,40,18
2,2025-03-09,BIHAR,MADHUBANI,18,120,22
3,2025-03-09,BIHAR,PURBI CHAMPARAN,48,120,22
4,2025-03-09,BIHAR,SITAMARHI,127,353,104
5,2025-03-09,DELHI,WEST DELHI,122,53,57
6,2025-03-09,HARYANA,FARIDABAD,80,60,10
7,2025-03-09,HARYANA,GURUGRAM,18,19,13
8,2025-03-09,KARNATAKA,BENGALURU URBAN,63,80,105
9,2025-03-09,MADHYA PRADESH,BHIND,37,18,11


In [78]:
print(enroll.shape)
print(enroll_dist.shape)

(1006029, 7)
(66487, 6)


In [79]:
# 2.2 Aggregate Aadhaar demographic update data

demo_dist = (
    demo
    .groupby(["date", "state", "district"], as_index=False)
    .agg({
        "demo_age_5_17": "sum",
        "demo_age_17_": "sum"
    })
)

In [81]:
demo_dist.head()

,date,state,district,demo_age_5_17,demo_age_17_
0,2025-03-01,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,32,360
1,2025-03-01,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,20,402
2,2025-03-01,ANDAMAN AND NICOBAR ISLANDS,SOUTH ANDAMAN,74,450
3,2025-03-01,ANDHRA PRADESH,ADILABAD,390,3950
4,2025-03-01,ANDHRA PRADESH,ALLURI SITHARAMA RAJU,507,4448


In [82]:
print(demo.shape)
print(demo_dist.shape)


(2071700, 6)
(85342, 5)


In [84]:
# 2.3 Aggregate Aadhaar biometric update data

bio_dist = (
    bio
    .groupby(["date", "state", "district"], as_index=False)
    .agg({
        "bio_age_5_17": "sum",
        "bio_age_17_": "sum"
    })
)

In [85]:
bio_dist.head()

,date,state,district,bio_age_5_17,bio_age_17_
0,2025-03-01,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,16,193
1,2025-03-01,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,178,101
2,2025-03-01,ANDAMAN AND NICOBAR ISLANDS,NORTH AND MIDDLE ANDAMAN,470,347
3,2025-03-01,ANDAMAN AND NICOBAR ISLANDS,SOUTH ANDAMAN,948,450
4,2025-03-01,ANDHRA PRADESH,ADILABAD,897,4366


In [86]:
print(bio.shape)
print(bio_dist.shape)

(1861108, 6)
(79179, 5)


In [ ]:
# 2.4 Validate aggregation correctness
assert "pincode" not in enroll_dist.columns
assert "pincode" not in demo_dist.columns
assert "pincode" not in bio_dist.columns


In [ ]:
# Checking if the totals are preserved

sample = enroll.query(
    "date == @enroll_dist.date.iloc[0] "
    "and state == @enroll_dist.state.iloc[0] "
    "and district == @enroll_dist.district.iloc[0]"
)

sample["age_0_5"].sum(), enroll_dist.iloc[0]["age_0_5"]


(np.int64(11), np.int64(11))

In [92]:
enroll_dist.to_csv("data/aggregated/enrolment_district_daily.csv", index=False)
demo_dist.to_csv("data/aggregated/demographic_district_daily.csv", index=False)
bio_dist.to_csv("data/aggregated/biometric_district_daily.csv", index=False)

### 3. Build the complete District–Date skeleton

In [ ]:
# 3.1 Creating a universal set of districts (Backbone for LEFT JOINs)
districts = pd.concat([
    enroll_dist[["state", "district"]],
    demo_dist[["state", "district"]],
    bio_dist[["state", "district"]],
]).drop_duplicates().reset_index(drop=True)


In [97]:
districts.shape


(1095, 2)

In [98]:
districts.head()

,state,district
0,MEGHALAYA,EAST KHASI HILLS
1,BIHAR,BHAGALPUR
2,BIHAR,MADHUBANI
3,BIHAR,PURBI CHAMPARAN
4,BIHAR,SITAMARHI


In [100]:
# 3.2 Create the global date range

min_date = min(
    enroll_dist["date"].min(),
    demo_dist["date"].min(),
    bio_dist["date"].min()
)

max_date = max(
    enroll_dist["date"].max(),
    demo_dist["date"].max(),
    bio_dist["date"].max()
)

all_dates = pd.date_range(start=min_date, end=max_date, freq="D")


In [102]:
len(all_dates), all_dates.min(), all_dates.max()

(306, Timestamp('2025-03-01 00:00:00'), Timestamp('2025-12-31 00:00:00'))

In [104]:
# 3.3 Build the cartesian product (the skeleton)

skeleton = (
    districts
    .assign(key=1)
    .merge(
        pd.DataFrame({"date": all_dates, "key": 1}),
        on="key"
    )
    .drop("key", axis=1)
    .sort_values(["date", "state", "district"])
    .reset_index(drop=True)
)

In [109]:
# 3.5 Validating the skeleton

# Check 1
expected_rows = len(districts) * len(all_dates)
assert skeleton.shape[0] == expected_rows

# Check 2 (Duplicates)

assert skeleton.duplicated(
    ["date", "state", "district"]
).sum() == 0

# Check 3 (Coverage)

skeleton[
    (skeleton["state"] == districts.iloc[0]["state"]) &
    (skeleton["district"] == districts.iloc[0]["district"])
].head(10)


,state,district,date
601,MEGHALAYA,EAST KHASI HILLS,2025-03-01
1696,MEGHALAYA,EAST KHASI HILLS,2025-03-02
2791,MEGHALAYA,EAST KHASI HILLS,2025-03-03
3886,MEGHALAYA,EAST KHASI HILLS,2025-03-04
4981,MEGHALAYA,EAST KHASI HILLS,2025-03-05
6076,MEGHALAYA,EAST KHASI HILLS,2025-03-06
7171,MEGHALAYA,EAST KHASI HILLS,2025-03-07
8266,MEGHALAYA,EAST KHASI HILLS,2025-03-08
9361,MEGHALAYA,EAST KHASI HILLS,2025-03-09
10456,MEGHALAYA,EAST KHASI HILLS,2025-03-10


In [110]:
skeleton.to_csv("data/district_date_skeleton.csv", index=False)


### 4. Join datasets onto the district–date skeleton

In [131]:
# 4.1 Left-join enrolment data

master_table = (
    skeleton
    .merge(
        enroll_dist,
        on=["date", "state", "district"],
        how="left"
    )
)


In [132]:
master_table.shape


(335070, 6)

In [133]:
master_table.head()

,state,district,date,age_0_5,age_5_17,age_18_greater
0,100000,100000,2025-03-01,NaN,NaN,NaN
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,NaN,NaN,NaN
2,ANDAMAN & NICOBAR ISLANDS,NICOBARS,2025-03-01,NaN,NaN,NaN
3,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,2025-03-01,NaN,NaN,NaN
4,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,2025-03-01,NaN,NaN,NaN


In [134]:
# 4.2 Left-join demographic data

master_table = master_table.merge(
    demo_dist,
    on=["date", "state", "district"],
    how="left"
)


In [135]:
master_table.shape

(335070, 8)

In [136]:
master_table.head()

,state,district,date,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_
0,100000,100000,2025-03-01,NaN,NaN,NaN,NaN,NaN
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,NaN,NaN,NaN,NaN,NaN
2,ANDAMAN & NICOBAR ISLANDS,NICOBARS,2025-03-01,NaN,NaN,NaN,NaN,NaN
3,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,2025-03-01,NaN,NaN,NaN,NaN,NaN
4,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,2025-03-01,NaN,NaN,NaN,32.0,360.0


In [ ]:
# 4.3 Left-join biometric update data

master_table = master_table.merge(
    bio_dist,
    on=["date", "state", "district"],
    how="left"
)


In [138]:
master_table.shape, master_table.isna().sum()

((335070, 10),
 state                  0
 district               0
 date                   0
 age_0_5           268583
 age_5_17          268583
 age_18_greater    268583
 demo_age_5_17     249728
 demo_age_17_      249728
 bio_age_5_17      255891
 bio_age_17_       255891
 dtype: int64)

In [145]:
# 4.4 Validate join correctness

# Check 1: Row count preserved
assert master_table.shape[0] == skeleton.shape[0]

# Check 2: No duplicate keys introduced
assert master_table.duplicated(
    ["date", "state", "district"]
).sum() == 0

# Check 3: Column list sanity
master_table.columns.to_list()



['state',
 'district',
 'date',
 'age_0_5',
 'age_5_17',
 'age_18_greater',
 'demo_age_5_17',
 'demo_age_17_',
 'bio_age_5_17',
 'bio_age_17_']

In [147]:
master_table.to_csv("data/aadhar_master_pre_fill.csv", index=False)


### 5. Apply ZERO logic using a single global start date

In [148]:
enroll_start = enroll_dist["date"].min()
demo_start   = demo_dist["date"].min()
bio_start    = bio_dist["date"].min()

enroll_start, demo_start, bio_start


(Timestamp('2025-03-02 00:00:00'),
 Timestamp('2025-03-01 00:00:00'),
 Timestamp('2025-03-01 00:00:00'))

In [ ]:
# 5. 1 Define the global start date
GLOBAL_START = pd.Timestamp("2025-03-01")

# 5. 2 Define measure columns
enroll_cols = [
    "age_0_5",
    "age_5_17",
    "age_18_greater"
]

demo_cols = [
    "demo_age_5_17",
    "demo_age_17_"
]

bio_cols = [
    "bio_age_5_17",
    "bio_age_17_"
]

ALL_MEASURE_COLS = enroll_cols + demo_cols + bio_cols


In [ ]:
# 5.3 Apply zero-filling from the global start date
master_table.loc[
    master_table["date"] >= GLOBAL_START,
    ALL_MEASURE_COLS
] = (
    master_table.loc[
        master_table["date"] >= GLOBAL_START,
        ALL_MEASURE_COLS
    ]
    .fillna(0)
)


In [ ]:
# 5.4 Validation checks

# Check 1: No NaNs after start date
assert (
    master_table.loc[master_table["date"] >= GLOBAL_START, ALL_MEASURE_COLS]
    .isna()
    .sum()
    .sum()
    == 0
)

# Check 2: Keys still unique
assert master_table.duplicated(["date", "state", "district"]).sum() == 0

In [158]:
# 5.5 add a single availability flag
master_table["data_available"] = master_table["date"] >= GLOBAL_START
master_table[master_table["data_available"]]

,state,district,date,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_,data_available
0,100000,100000,2025-03-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,0.0,0.0,0.0,0.0,0.0,16.0,193.0,True
2,ANDAMAN & NICOBAR ISLANDS,NICOBARS,2025-03-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
3,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,2025-03-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
4,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,2025-03-01,0.0,0.0,0.0,32.0,360.0,178.0,101.0,True
...,...,...,...,...,...,...,...,...,...,...,...
335065,WEST BENGAL,WEST MEDINIPUR,2025-12-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
335066,WEST BENGAL,WEST MIDNAPORE,2025-12-31,22.0,20.0,1.0,0.0,0.0,0.0,0.0,True
335067,WEST BENGLI,HOOGHLY,2025-12-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
335068,WESTBENGAL,HOOGHLY,2025-12-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True


In [157]:
master_table.to_csv("data/aadhaar_master_table.csv", index=False)
